# Hospital Revenue — Data Quality Report

Notebook kiểm tra chất lượng dữ liệu theo 4 nhóm:
1. **Missing Values** — Giá trị thiếu theo cột và theo thời gian
2. **Outlier Detection** — Phát hiện giao dịch bất thường (IQR + Z-score)
3. **Duplicate Detection** — Phát hiện giao dịch trùng lặp
4. **Data Validation** — Giá trị âm, bằng 0, logic không hợp lệ
5. **Export** — Xuất kết quả ra JSON cho dashboard

> Sau khi chạy xong, vào `http://127.0.0.1:5000/data-quality` để xem báo cáo.

## Section 1 — Load Data

In [1]:
import sqlite3
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

DB_PATH    = Path("../backend/hospital.db")
OUTPUT_DIR = Path("../backend/analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True)

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
    SELECT
        f.ID, f.NGAY,
        f.ID_BENHNHAN, f.ID_KHAMBENH,
        f.ID_KHOAPHONG, f.ID_DICHVU, f.ID_DOITUONG, f.ID_LOAI_DIEUTRI,
        f.SOLUONG, f.DONGIA, f.THANHTIEN, f.BHYTTRA, f.NGUOIBENHTRA,
        k.TEN_KHOAPHONG,
        d.TEN_DICHVU,
        dt.TEN_DOITUONG
    FROM FACT_DOANHTHU f
    LEFT JOIN DIM_KHOAPHONG k   ON f.ID_KHOAPHONG = k.ID_KHOAPHONG
    LEFT JOIN DIM_DICHVU d      ON f.ID_DICHVU    = d.ID_DICHVU
    LEFT JOIN DIM_DOITUONG dt   ON f.ID_DOITUONG  = dt.ID_DOITUONG
""", conn)
conn.close()

df["NGAY"]     = pd.to_datetime(df["NGAY"])
df["THANG_NAM"] = df["NGAY"].dt.to_period("M").astype(str)

print(f"Tổng số bản ghi : {len(df):,}")
print(f"Khoảng thời gian: {df['NGAY'].min().date()} → {df['NGAY'].max().date()}")
df.head()

Tổng số bản ghi : 7,233
Khoảng thời gian: 2026-01-01 → 2026-05-10


,ID,NGAY,ID_BENHNHAN,ID_KHAMBENH,ID_KHOAPHONG,ID_DICHVU,ID_DOITUONG,ID_LOAI_DIEUTRI,SOLUONG,DONGIA,THANHTIEN,BHYTTRA,NGUOIBENHTRA,TEN_KHOAPHONG,TEN_DICHVU,TEN_DOITUONG,THANG_NAM
0,1,2026-01-01,BN00139,KB202601019817,8,8,1,2,2,8334897.0,16669794.0,6667918.0,10001877.0,Khoa Sản,Phẫu thuật loại 1,BHYT,2026-01
1,2,2026-01-01,BN00045,KB202601015569,8,4,2,2,3,89527.0,268582.0,0.0,268582.0,Khoa Sản,Xét nghiệm nước tiểu,Dịch vụ,2026-01
2,3,2026-01-01,BN00015,KB202601019098,6,3,1,1,2,118210.0,236420.0,189136.0,47284.0,Khoa Chẩn đoán hình ảnh,Xét nghiệm máu,BHYT,2026-01
3,4,2026-01-01,BN00142,KB202601011166,9,4,1,2,2,93491.0,186983.0,74793.0,112190.0,Khoa Nhi,Xét nghiệm nước tiểu,BHYT,2026-01
4,5,2026-01-01,BN00244,KB202601019041,8,1,1,1,3,144124.0,432372.0,345897.0,86474.0,Khoa Sản,Khám bệnh thường,BHYT,2026-01


## Section 2 — Missing Values Analysis

In [2]:
CHECK_COLS = {
    "ID_KHOAPHONG" : "Khoa/phòng",
    "ID_DICHVU"    : "Dịch vụ",
    "ID_DOITUONG"  : "Đối tượng TT",
    "ID_BENHNHAN"  : "Bệnh nhân",
    "ID_KHAMBENH"  : "Mã lượt khám",
}

total = len(df)
missing_by_col = []
for col, label in CHECK_COLS.items():
    n = int(df[col].isnull().sum()) if col in df.columns else 0
    missing_by_col.append({
        "column" : col,
        "label"  : label,
        "n_missing": n,
        "pct"    : round(n / total * 100, 2) if total else 0,
    })

# Monthly missing trend
monthly_missing = (
    df.groupby("THANG_NAM")
    .apply(lambda g: pd.Series({
        "total"             : len(g),
        "missing_khoaphong" : int(g["ID_KHOAPHONG"].isnull().sum()),
        "missing_dichvu"    : int(g["ID_DICHVU"].isnull().sum()),
        "missing_doituong"  : int(g["ID_DOITUONG"].isnull().sum()),
    }))
    .reset_index()
)
monthly_missing["pct_missing"] = (
    (monthly_missing[["missing_khoaphong","missing_dichvu","missing_doituong"]].sum(axis=1)
     / monthly_missing["total"] * 100)
    .round(2)
)

print("Missing values theo cột:")
for m in missing_by_col:
    status = "✅" if m["n_missing"] == 0 else "⚠️"
    print(f"  {status} {m['label']:20s}: {m['n_missing']:>6,} ({m['pct']:.1f}%)")

Missing values theo cột:
  ✅ Khoa/phòng          :      0 (0.0%)
  ✅ Dịch vụ             :      0 (0.0%)
  ✅ Đối tượng TT        :      0 (0.0%)
  ✅ Bệnh nhân           :      0 (0.0%)
  ✅ Mã lượt khám        :      0 (0.0%)


/var/folders/wn/ylz0hq2s07v8kffwzxltstrm0000gn/T/ipykernel_94483/935734416.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("THANG_NAM")


In [3]:
# Bar chart missing values
fig = px.bar(
    pd.DataFrame(missing_by_col),
    x="label", y="pct",
    title="Tỷ lệ missing values theo cột (%)",
    labels={"label": "Cột", "pct": "% thiếu"},
    color="pct",
    color_continuous_scale=["#43a047", "#fb8c00", "#e53935"],
    text_auto=".1f",
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

# Monthly missing trend
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=monthly_missing["THANG_NAM"], y=monthly_missing["pct_missing"],
    mode="lines+markers", name="% missing",
    line=dict(color="#fb8c00", width=2),
))
fig2.update_layout(
    title="Xu hướng tỷ lệ missing values theo tháng",
    xaxis_title="Tháng", yaxis_title="% missing",
)
fig2.show()

## Section 3 — Outlier Detection

In [4]:
# ── IQR method ───────────────────────────────────────────────────────
Q1  = df["THANHTIEN"].quantile(0.25)
Q3  = df["THANHTIEN"].quantile(0.75)
IQR = Q3 - Q1
lower_iqr = Q1 - 1.5 * IQR
upper_iqr = Q3 + 1.5 * IQR

mask_iqr     = (df["THANHTIEN"] < lower_iqr) | (df["THANHTIEN"] > upper_iqr)
outliers_iqr = df[mask_iqr].copy()

# ── Z-score method ───────────────────────────────────────────────────
df["ZSCORE"] = np.abs(stats.zscore(df["THANHTIEN"].fillna(0)))
outliers_z   = df[df["ZSCORE"] > 3].copy()

# ── Union of both methods ────────────────────────────────────────────
outlier_ids  = set(outliers_iqr["ID"].tolist()) | set(outliers_z["ID"].tolist())
outliers_all = df[df["ID"].isin(outlier_ids)].copy()

print(f"IQR bounds: [{lower_iqr:,.0f} – {upper_iqr:,.0f}] VNĐ")
print(f"Outliers IQR    : {len(outliers_iqr):,} ({len(outliers_iqr)/total*100:.1f}%)")
print(f"Outliers Z-score: {len(outliers_z):,} ({len(outliers_z)/total*100:.1f}%)")
print(f"Outliers (union): {len(outliers_all):,} ({len(outliers_all)/total*100:.1f}%)")

IQR bounds: [-1,328,590 – 2,940,194] VNĐ
Outliers IQR    : 1,153 (15.9%)
Outliers Z-score: 286 (4.0%)
Outliers (union): 1,153 (15.9%)


In [5]:
# Box plot phát hiện outlier
fig = go.Figure()
fig.add_trace(go.Box(
    y=df["THANHTIEN"], name="Doanh thu/GD",
    boxpoints="outliers", marker_color="#1976d2",
    line_color="#1565c0",
))
fig.update_layout(
    title="Phân phối THANHTIEN — Box Plot (điểm đỏ = outlier)",
    yaxis_title="Thành tiền (VNĐ)",
)
fig.show()

# Scatter outliers theo thời gian
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df["NGAY"], y=df["THANHTIEN"],
    mode="markers", name="Bình thường",
    marker=dict(color="#90a4ae", size=3, opacity=0.5),
))
fig2.add_trace(go.Scatter(
    x=outliers_all["NGAY"], y=outliers_all["THANHTIEN"],
    mode="markers", name="Outlier",
    marker=dict(color="#e53935", size=7, symbol="x"),
))
fig2.update_layout(
    title="Outlier theo thời gian",
    xaxis_title="Ngày", yaxis_title="Thành tiền (VNĐ)",
)
fig2.show()

## Section 4 — Duplicate Detection

In [6]:
DUP_COLS = ["NGAY", "ID_BENHNHAN", "ID_KHOAPHONG", "ID_DICHVU"]

dup_mask   = df.duplicated(subset=DUP_COLS, keep=False)
duplicates = df[dup_mask].sort_values(DUP_COLS).copy()
n_dup_rows = len(duplicates)
# Number of duplicate groups (records beyond the first in each group)
n_dup_extra = int(df.duplicated(subset=DUP_COLS, keep="first").sum())

print(f"Số bản ghi thuộc nhóm trùng lặp: {n_dup_rows:,}")
print(f"Số bản ghi dư thừa (cần xem xét): {n_dup_extra:,} ({n_dup_extra/total*100:.2f}%)")

if n_dup_rows > 0:
    display(duplicates[["ID","NGAY","ID_BENHNHAN","TEN_KHOAPHONG","TEN_DICHVU","SOLUONG","DONGIA","THANHTIEN"]].head(20))

Số bản ghi thuộc nhóm trùng lặp: 6
Số bản ghi dư thừa (cần xem xét): 3 (0.04%)


,ID,NGAY,ID_BENHNHAN,TEN_KHOAPHONG,TEN_DICHVU,SOLUONG,DONGIA,THANHTIEN
117,118,2026-01-03,BN00174,Khoa Sản,Vật tư y tế,1,228944.0,228944.0
121,122,2026-01-03,BN00174,Khoa Sản,Vật tư y tế,3,241670.0,725010.0
524,525,2026-01-12,BN00058,Khoa Dược,Siêu âm,3,204581.0,613742.0
558,559,2026-01-12,BN00058,Khoa Dược,Siêu âm,3,234048.0,702143.0
5165,5166,2026-04-12,BN00272,Khoa Sản,Ngày giường nội trú,3,464013.0,1392040.0
5201,5202,2026-04-12,BN00272,Khoa Sản,Ngày giường nội trú,2,533736.0,1067473.0


## Section 5 — Data Validation

In [7]:
# Negative revenue
neg_rev  = df[df["THANHTIEN"] < 0]
# Zero revenue
zero_rev = df[df["THANHTIEN"] == 0]
# Negative unit price
neg_price = df[df["DONGIA"] < 0]
# Quantity <= 0
zero_qty  = df[df["SOLUONG"] <= 0]
# BHYT > THANHTIEN (không hợp lệ)
bhyt_over = df[df["BHYTTRA"] > df["THANHTIEN"]]
# Unmapped dimension (dimension name is null but ID exists)
unmapped_khoa  = df[(df["ID_KHOAPHONG"].notna()) & (df["TEN_KHOAPHONG"].isnull())]
unmapped_dichvu= df[(df["ID_DICHVU"].notna())    & (df["TEN_DICHVU"].isnull())]
unmapped_dt    = df[(df["ID_DOITUONG"].notna())  & (df["TEN_DOITUONG"].isnull())]

validation_results = [
    {"check": "Doanh thu âm",            "count": len(neg_rev),    "pct": round(len(neg_rev)/total*100,2)},
    {"check": "Doanh thu bằng 0",         "count": len(zero_rev),   "pct": round(len(zero_rev)/total*100,2)},
    {"check": "Đơn giá âm",              "count": len(neg_price),  "pct": round(len(neg_price)/total*100,2)},
    {"check": "Số lượng <= 0",           "count": len(zero_qty),   "pct": round(len(zero_qty)/total*100,2)},
    {"check": "BHYT trả > Thành tiền",   "count": len(bhyt_over),  "pct": round(len(bhyt_over)/total*100,2)},
    {"check": "Khoa không map được",      "count": len(unmapped_khoa),  "pct": round(len(unmapped_khoa)/total*100,2)},
    {"check": "Dịch vụ không map được",   "count": len(unmapped_dichvu),"pct": round(len(unmapped_dichvu)/total*100,2)},
    {"check": "Đối tượng không map được", "count": len(unmapped_dt),    "pct": round(len(unmapped_dt)/total*100,2)},
]

print("Kết quả Data Validation:")
for v in validation_results:
    icon = "✅" if v["count"] == 0 else "❌"
    print(f"  {icon} {v['check']:35s}: {v['count']:>6,} ({v['pct']:.2f}%)")

Kết quả Data Validation:
  ✅ Doanh thu âm                       :      0 (0.00%)
  ✅ Doanh thu bằng 0                   :      0 (0.00%)
  ✅ Đơn giá âm                         :      0 (0.00%)
  ✅ Số lượng <= 0                      :      0 (0.00%)
  ✅ BHYT trả > Thành tiền              :      0 (0.00%)
  ✅ Khoa không map được                :      0 (0.00%)
  ✅ Dịch vụ không map được             :      0 (0.00%)
  ✅ Đối tượng không map được           :      0 (0.00%)


In [8]:
vdf = pd.DataFrame(validation_results)
fig = px.bar(
    vdf[vdf["count"] > 0] if vdf["count"].sum() > 0 else vdf,
    x="check", y="count",
    title="Số bản ghi vi phạm quy tắc validation",
    labels={"check": "Quy tắc", "count": "Số bản ghi"},
    color="count",
    color_continuous_scale=["#43a047", "#fb8c00", "#e53935"],
    text_auto=True,
)
fig.update_layout(coloraxis_showscale=False, xaxis_tickangle=-20)
fig.show()

## Section 6 — Overall Quality Score & Export

In [9]:
# ── Quality Score ────────────────────────────────────────────────────
total_missing  = sum(m["n_missing"] for m in missing_by_col)
total_outliers = len(outliers_all)
total_dups     = n_dup_extra
total_invalid  = sum(v["count"] for v in validation_results)

# Weighted: missing & invalid count full, outliers & dups count half
weighted_issues = total_missing + total_invalid + (total_outliers + total_dups) * 0.5
quality_score   = max(0.0, (1 - weighted_issues / total) * 100) if total > 0 else 100.0

# ── Auto recommendations ─────────────────────────────────────────────
recs = []
if total_missing > 0:
    recs.append(f"Có {total_missing:,} giá trị thiếu cần điền bổ sung hoặc kiểm tra quy trình nhập liệu.")
if total_outliers > 0:
    recs.append(f"Phát hiện {total_outliers:,} giao dịch bất thường — xác nhận với bộ phận viện phí.")
if total_dups > 0:
    recs.append(f"Có {total_dups:,} bản ghi có khả năng trùng lặp — rà soát để tránh tính phí 2 lần.")
if total_invalid > 0:
    recs.append(f"Có {total_invalid:,} bản ghi vi phạm quy tắc validation cần sửa.")
if not recs:
    recs.append("Dữ liệu đạt chất lượng tốt. Tiếp tục duy trì quy trình kiểm soát hiện tại.")

print(f"\n📊 Quality Score: {quality_score:.1f}/100")
print(f"   Missing   : {total_missing:,}")
print(f"   Invalid   : {total_invalid:,}")
print(f"   Outliers  : {total_outliers:,}")
print(f"   Duplicates: {total_dups:,}")

# ── Export ───────────────────────────────────────────────────────────
generated_at = datetime.datetime.now().isoformat()

outlier_samples = (
    outliers_all
    .sort_values("THANHTIEN", ascending=False)
    .head(20)[["ID","NGAY","ID_BENHNHAN","TEN_KHOAPHONG","TEN_DICHVU","THANHTIEN","ZSCORE"]]
    .assign(NGAY=lambda d: d["NGAY"].dt.strftime("%Y-%m-%d"))
    .fillna("—")
    .round(2)
    .to_dict(orient="records")
)

dup_samples = (
    duplicates
    .head(20)[["ID","NGAY","ID_BENHNHAN","TEN_KHOAPHONG","TEN_DICHVU","SOLUONG","THANHTIEN"]]
    .assign(NGAY=lambda d: d["NGAY"].dt.strftime("%Y-%m-%d"))
    .fillna("—")
    .to_dict(orient="records")
)

dq_output = {
    "generated_at"   : generated_at,
    "total_records"  : total,
    "quality_score"  : round(quality_score, 1),
    "summary": {
        "total_missing" : total_missing,
        "total_outliers": total_outliers,
        "total_duplicates": total_dups,
        "total_invalid" : total_invalid,
    },
    "missing_by_column": missing_by_col,
    "missing_trend"    : monthly_missing.to_dict(orient="records"),
    "outliers": {
        "iqr_lower" : round(lower_iqr, 0),
        "iqr_upper" : round(upper_iqr, 0),
        "count"     : total_outliers,
        "pct"       : round(total_outliers / total * 100, 2) if total else 0,
        "samples"   : outlier_samples,
    },
    "duplicates": {
        "count"  : total_dups,
        "pct"    : round(total_dups / total * 100, 2) if total else 0,
        "samples": dup_samples,
    },
    "validation"     : validation_results,
    "recommendations": recs,
}

out = OUTPUT_DIR / "data_quality_report.json"
out.write_text(json.dumps(dq_output, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"\n✅ {out}")
print(f"   Dashboard: http://127.0.0.1:5000/data-quality")


📊 Quality Score: 92.0/100
   Missing   : 0
   Invalid   : 0
   Outliers  : 1,153
   Duplicates: 3

✅ ../backend/analysis_output/data_quality_report.json
   Dashboard: http://127.0.0.1:5000/data-quality
